# RunnableWithMessageHistory — LangChain 1.0.x 공식 메모리 패턴

## 이번 노트북 학습 목표

- `RunnableWithMessageHistory`(실행 가능한 메시지 히스토리 래퍼)가 왜 필요한지 이해해요.
- `get_session_history` 함수를 만들어 세션별 대화 기록을 관리하는 방법을 익혀요.
- `config={"configurable": {"session_id": "..."}}` 패턴으로 세션을 전환하는 방법을 확인해요.
- 인메모리 백엔드와 SQLite 백엔드를 비교해요.

## 사전 지식

- LCEL 파이프라인 기본 (08번 노트북)
- `SQLChatMessageHistory` 사용법 (09번 노트북)

## 08번(수동)과 이번 노트북(자동)의 차이

| 항목 | 08번 (수동 LCEL) | 10번 (RunnableWithMessageHistory) |
|------|-----------------|-----------------------------------|
| 메모리 저장 | `memory.save_context()` 직접 호출 | `invoke()` 시 자동 처리 |
| 세션 분리 | 별도 구현 필요 | `session_id`만 바꾸면 됨 |
| 백엔드 교체 | 코드 전체 수정 | 함수 한 곳만 수정 |
| LangChain 권장 | 과도기적 패턴 | 1.0.x 공식 권장 방식 |


In [ ]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()


## 1. 기본 LCEL 체인 생성

`RunnableWithMessageHistory`는 기존 LCEL 체인을 감싸는(wrap) 래퍼예요. 먼저 메모리 없는 기본 체인을 만들고, 그 위에 메모리 래퍼를 씌우는 순서로 진행해요.


### RunnableWithMessageHistory 전체 흐름

```mermaid
flowchart LR
    USER[사용자 입력<br/>+ session_id] --> RWMH[RunnableWithMessageHistory]
    RWMH --> LOAD[세션 히스토리 로드]
    LOAD --> PROMPT[ChatPromptTemplate<br/>chat_history 채워짐]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> PARSER[StrOutputParser]
    PARSER --> SAVE[세션 히스토리 저장<br/>자동]
    SAVE --> ANSWER[AI 응답 반환]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class USER input
    class RWMH,LOAD,PROMPT,LLM,PARSER,SAVE process
    class ANSWER output
```


In [ ]:
# ---------------------------------------------------
# LCEL 기본 체인 구성 (메모리 없는 베이스 체인)
# ---------------------------------------------------

# 1단계: 프롬프트 정의
# - MessagesPlaceholder: chat_history 변수명으로 대화 이력이 주입됨
# - 가급적 variable_name="chat_history"를 변경하지 말 것 (하위 코드와 일치 필요)
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 친절한 AI 어시스턴트입니다. 사용자의 질문에 대해 도움을 제공해주세요.",
    ),
    # 대화 기록용 key인 chat_history는 가급적 변경 없이 사용하세요!
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),  # 사용자 입력을 변수로 사용
])

# 2단계: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini")

# 3단계: 기본 Chain 생성 (메모리 없음)
# - 이 단계에서는 아직 메모리가 없음. 아래 RunnableWithMessageHistory로 감쌀 예정
chain = prompt | llm | StrOutputParser()

## 2. 세션 관리 함수와 RunnableWithMessageHistory

`get_session_history` 함수는 `session_id`를 받아 해당 세션의 `ChatMessageHistory` 객체를 반환해요. 처음 보는 `session_id`라면 새로 만들고, 이미 있으면 기존 것을 반환해요.

### 세션 히스토리 조회 로직

```mermaid
flowchart TD
    SESSION_ID[session_id] --> CHECK{store에<br/>이미 있나요?}
    CHECK -- 예 --> RETURN[기존 ChatMessageHistory 반환]
    CHECK -- 아니오 --> CREATE[새 ChatMessageHistory 생성]
    CREATE --> STORE_UPDATE[store에 저장]
    STORE_UPDATE --> RETURN

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class SESSION_ID input
    class CHECK,CREATE process
    class STORE_UPDATE storage
    class RETURN output
```


In [ ]:
# ============================================================
# TODO: 세션 히스토리 관리 함수와 RunnableWithMessageHistory를 구성하세요
# 힌트: store 딕셔너리로 세션별 ChatMessageHistory를 관리하고,
#       RunnableWithMessageHistory로 chain을 감싸세요
# 예상 결과: session_id가 같으면 이전 대화를 기억하고, 다르면 새 대화로 시작해야 합니다
# ============================================================

# ---------------------------------------------------
# 세션 스토어와 히스토리 관리 함수 구성
# ---------------------------------------------------

# 1단계: 세션별 대화 기록을 저장할 인메모리 스토어
# - 딕셔너리 키: session_id, 값: ChatMessageHistory 객체
store = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    """세션 ID를 기반으로 대화 기록을 가져오는 함수.

    - 없으면 새 ChatMessageHistory를 생성해서 store에 등록
    - 있으면 기존 히스토리를 그대로 반환
    """
    if session_id not in store:
        # 처음 보는 세션 ID → 새 히스토리 생성
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 2단계: RunnableWithMessageHistory로 체인 감싸기
# - chain: 기존 LCEL 기본 체인 (위에서 정의한 prompt | llm | StrOutputParser())
# - input_messages_key: 프롬프트의 사용자 입력 변수명 ({question})
# - history_messages_key: MessagesPlaceholder의 variable_name (chat_history)


In [ ]:
# ---------------------------------------------------
# 세션 대화 실행: 동일 세션은 이전 내용을 기억
# ---------------------------------------------------

# 1단계: user_001 세션 — 첫 번째 대화 (자기소개)
# - config의 session_id로 어떤 세션인지 지정


# 2단계: user_001 세션 — 두 번째 대화 (이전 내용 기억 여부 확인)


# 3단계: user_002 세션 — 완전히 다른 독립 세션
# - 다른 session_id 사용 → 이전 대화를 알 수 없음


### 💡 새로운 세션(user_002) 흐름

```mermaid
flowchart TD
    NEW_ID[user_002] --> SESSION_FN
    SESSION_FN --> NEW_HISTORY[새 ChatMessageHistory 생성]
    NEW_HISTORY --> AUG_CHAIN
    AUG_CHAIN --> RESPONSE_NEW[기억 없음 응답]
```



## 핵심 요약

### RunnableWithMessageHistory 핵심 파라미터

| 파라미터 | 역할 |
|---------|------|
| `runnable` | 기존 LCEL 체인 |
| `get_session_history` | `session_id` → `ChatMessageHistory` 반환 함수 |
| `input_messages_key` | 프롬프트의 사용자 입력 변수명 (`{question}`) |
| `history_messages_key` | 프롬프트의 `MessagesPlaceholder` 변수명 |

### 주의사항

- `history_messages_key`와 `MessagesPlaceholder(variable_name=...)` 값이 동일해야 해요.
- 개발·테스트 환경에서는 `dict` 스토어를 사용하고, 프로덕션에서는 SQLite 이상의 DB를 권장해요.
- `session_id`는 예측 불가능한 값(UUID 등)을 사용해야 사용자 간 대화 혼재를 막을 수 있어요.

### 다음 단계 예고

**11번**: Legacy 메모리 클래스와 Modern 패턴의 전체 매핑 테이블을 살펴봐요. `ConversationBufferWindowMemory`, `ConversationSummaryMemory` 등이 새 패턴에서 어떻게 재설계되는지 이해해요.


---

## 연습 문제

### 연습 1: 역할별 시스템 프롬프트를 가진 다중 세션 챗봇

`RunnableWithMessageHistory`를 활용하여, **세션 ID에 따라 다른 시스템 프롬프트**를 적용하는 챗봇을 만들어 보세요.

**요구사항:**
- `"tutor_session"` 세션: 시스템 프롬프트를 "당신은 친절한 프로그래밍 튜터입니다. 코드 예제와 함께 쉽게 설명해주세요."로 설정
- `"chef_session"` 세션: 시스템 프롬프트를 "당신은 한식 전문 요리사입니다. 레시피와 요리 팁을 알려주세요."로 설정
- 각 세션에 2개의 질문을 보내고, 세션별로 다른 역할의 응답이 나오는지 확인하세요

**힌트:**
- 세션 ID에 따라 다른 프롬프트를 사용하는 체인을 만드세요
- `get_session_history` 함수는 기존과 동일하게 사용할 수 있습니다
- 각 역할별로 별도의 `RunnableWithMessageHistory`를 만들거나, 하나의 체인에서 분기하는 방법을 사용할 수 있습니다